# Transformers from the Ground Up

## Building Intuition One Step at a Time

This notebook builds the **complete mental model for the Transformer architecture** from first principles - starting with a tiny 3-dimensional semantic space that you can visualise, rotate, and reason about concretely.

Every concept is demonstrated on the same running example:

> **"the cat sat on the mat"**

| Step | Concept | Key Idea |
| ---- | ------- | -------- |
| 1  | Vocabulary + 3D Embeddings     | Words as points in semantic space |
| 2  | The Ordering Problem           | Why bags of words lose meaning |
| 3  | Sinusoidal PE                  | Adding position with sine waves |
| 4  | RoPE                           | Rotating Q/K vectors for relative position |
| 5  | Q, K, V + Attention            | Soft dictionary lookup |
| 6  | Multi-Head Attention           | Parallel attention heads |
| 7  | Feed-Forward + Layer Norm      | Per-token transformation + stabilisation |
| 8  | Full Transformer Block         | All components assembled |
| 9  | Mini Language Model            | End-to-end training from scratch |
| 10 | W_V as Relevance Filter        | What each token contributes to the blend |
| 11 | Causal Triangle                | Layer stacking and last-position richness |
| 12 | Encoder Architecture           | Bidirectional attention - mask=None |
| 13 | Cross-Attention                | Q from decoder, K/V from encoder |
| 14 | Encoder-Decoder                | Reversal task with cross-attention |
| 15 | Architecture Comparison        | Reader, Writer, Translator side by side |
| 16 | GPT-2 Internals                | A real model, cracked open |

**Scope note:** this notebook builds real, measured intuition for the full mechanism space above; a few adjacent engineering topics (KV-caching, other positional-encoding schemes, decoding variants) are named but not built - see the full tier breakdown in *What This Notebook Covered* near the end.


![The seven-step learning journey through this notebook, from raw tokens to a live pretrained Transformer](images/transformer-learning-journey.png)

---

## Prerequisite Bridge — From `01-rnns` DL Foundations

| Foundation (from `01-rnns/PT-Part1-Intro.ipynb`) | Role in this notebook |
|---|---|
| `nn.Module` subclassing, explicit training loop | Every mechanism here (`MultiHeadAttention`, `FeedForward`, `TransformerBlock`) is an `nn.Module`; training uses the identical 4-step loop |
| Computation graph and `.backward()` | Gradients flow through Q·Kᵀ/√dₖ→softmax→V; the same chain-rule traced on `y=x²` in `01-rnns` now runs through 100M-parameter attention layers |
| Autograd warm-up (`y = x²`, parabola minimisation) | The gradient-descent intuition built on the toy parabola carries over directly to the loss surface here |
| Tensor shapes: `(batch, features)` → `(batch, seq, dim)` | The shape contract expands by one axis; every `(B, S, D)` shape annotation in this notebook assumes comfort with the `01-rnns` shape exercises |

> **If you haven't run `01-rnns/PT-Part1-Intro.ipynb`** the autograd and shape-reasoning patterns used here will feel unfamiliar. Complete that notebook first — it takes under 30 minutes.

In [ ]:
#  Install dependencies (run once) 
import subprocess, sys

required = [
    ("numpy",        "numpy"),
    ("matplotlib",   "matplotlib"),
    ("torch",        "torch"),
    ("seaborn",      "seaborn"),
    ("plotly",       "plotly"),
    ("transformers", "transformers"),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")


In [ ]:
#  Imports 
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

warnings.filterwarnings('ignore')

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})
print(f"torch   {torch.__version__}")
print(f"numpy   {np.__version__}")


---

## Part 1 - Our Mini Universe: Vocabulary & Embeddings

Before anything else, we need a way to represent words as numbers. This is the job of an **embedding**.

In a real model (e.g. GPT-2), each word lives in a 768-dimensional space. Instead we build a **3-dimensional** semantic space where each axis captures a meaningful property:

| Axis | Meaning | Low (0) | High (1) |
| ---- | ------- | ------- | -------- |
| **dim 0** | Concreteness | abstract (articles) | physical objects (cat, mat) |
| **dim 1** | Animacy | inanimate (mat, fence) | living beings (cat, dog) |
| **dim 2** | Dynamism | static (mat, fence) | action words (ran, jumped) |


In [ ]:
#  Vocabulary 
VOCAB = {
    "<PAD>": 0, "<BOS>": 1, "<EOS>": 2,
    "the": 3, "a": 4, "cat": 5, "dog": 6,
    "mat": 7, "fence": 8, "sat": 9, "ran": 10,
    "jumped": 11, "on": 12, "over": 13, "big": 14,
}
IDX2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

#  3D Semantic Embeddings (Concreteness, Animacy, Dynamism) 
E = {
    "<PAD>": [0.00, 0.00, 0.00], "<BOS>": [0.08, 0.08, 0.15], "<EOS>": [0.08, 0.08, 0.15],
    "the":   [0.05, 0.04, 0.08], "a":     [0.05, 0.04, 0.08],
    "cat":   [0.91, 0.94, 0.38], "dog":   [0.88, 0.92, 0.55],
    "mat":   [0.96, 0.04, 0.04], "fence": [0.93, 0.03, 0.03],
    "sat":   [0.34, 0.18, 0.78], "ran":   [0.28, 0.12, 0.96],
    "jumped":[0.30, 0.14, 0.98], "on":    [0.14, 0.04, 0.18],
    "over":  [0.17, 0.04, 0.24], "big":   [0.44, 0.04, 0.09],
}

embedding_matrix = torch.tensor(
    [E[IDX2WORD[i]] for i in range(VOCAB_SIZE)], dtype=torch.float32
)

SENTENCE = "the cat sat on the mat"
TOKENS = SENTENCE.split()
TOKEN_IDS = [VOCAB[w] for w in TOKENS]
SEQ_LEN = len(TOKENS)

print(f"Vocab size       : {VOCAB_SIZE}")
print(f"Embedding shape  : {tuple(embedding_matrix.shape)}  (vocab x 3D)")
print(f"Running sentence : {SENTENCE!r}")
print(f"Token IDs        : {TOKEN_IDS}")
print()
print("Embedding matrix -> Concreteness, Animacy, Dynamism:")
for word, vec in list(E.items())[3:]:
    print(f"  {word:<10} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")


In [ ]:
#  Interactive 3D Vocabulary Visualisation 
# Plotly = interactive (drag to rotate). Matplotlib = static fallback.

CATEGORIES = {
    "Article":        (["the", "a"],            "#636EFA"),
    "Animate Noun":   (["cat", "dog"],           "#00CC96"),
    "Inanimate Noun": (["mat", "fence"],         "#AB63FA"),
    "Verb":           (["sat", "ran", "jumped"], "#EF553B"),
    "Preposition":    (["on", "over"],           "#FFA15A"),
    "Adjective":      (["big"],                  "#19D3F3"),
}

if HAS_PLOTLY:
    fig = go.Figure()
    for cat, (words, color) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='markers+text', text=words,
            textposition='top center', name=cat,
            marker=dict(size=10, color=color, opacity=0.85, line=dict(color='white', width=1))))
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    fig.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='lines', name='Sentence path',
        line=dict(color='gold', width=4, dash='dot')))
    fig.update_layout(
        title=dict(text='<b>3D Semantic Embedding Space</b> - drag to rotate', x=0.5),
        scene=dict(xaxis_title='Concreteness', yaxis_title='Animacy', zaxis_title='Dynamism'),
        width=820, height=560)
    fig.show()
else:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    cmap = {'Article': 'royalblue', 'Animate Noun': 'mediumseagreen',
            'Inanimate Noun': 'mediumpurple', 'Verb': 'tomato',
            'Preposition': 'darkorange', 'Adjective': 'deepskyblue'}
    for cat, (words, _) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        ax.scatter(xs, ys, zs, s=90, label=cat, color=cmap[cat], alpha=0.9)
        for w in words:
            ax.text(E[w][0], E[w][1], E[w][2], f' {w}', fontsize=9)
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    ax.plot(sx, sy, sz, 'o--', color='gold', lw=2, label='Sentence path')
    ax.set_xlabel('Concreteness'); ax.set_ylabel('Animacy'); ax.set_zlabel('Dynamism')
    ax.set_title('3D Semantic Embedding Space'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print('Tip: pip install plotly for an interactive, rotatable version')


### Tokenisation

A **tokeniser** converts a raw string into integer IDs the model can process. In our toy system, one word = one token. Production models use sub-word tokenisation (BPE / SentencePiece).


In [ ]:
#  Tokeniser 
def encode(text: str, add_bos: bool = False, add_eos: bool = False):
    ids = [VOCAB.get(w, VOCAB["<PAD>"]) for w in text.lower().split()]
    if add_bos:
        ids = [VOCAB["<BOS>"]] + ids
    if add_eos:
        ids = ids + [VOCAB["<EOS>"]]
    return ids


def decode(ids):
    return " ".join(IDX2WORD.get(i, "<?>") for i in ids)


phrase = "the big cat jumped over the fence"
enc = encode(phrase)
dec = decode(enc)
print(f"Input  : {phrase!r}")
print(f"Encoded: {enc}")
print(f"Decoded: {dec!r}")
print()
print('With BOS/EOS markers:')
enc2 = encode(phrase, add_bos=True, add_eos=True)
print(f"  {enc2}")
print()
print("  -> One word = one token; BPE splits rare words in real models.")


---

## Attention: First Contact

Before positions, before $Q/K/V$ projections, before multi-head - the beating heart of the transformer is one simple idea:

> **Every token looks at every other token and builds a weighted average of them, where the weights answer "how much do I care about you?"**

**Running it on our sentence:** `"the cat sat on the mat"`. Query = `"cat"`. No positions, no learned projections - just raw dot products of the 3D semantic vectors.


In [ ]:
#  Attention, step by step - freezing at each completed step 
# Minimal attention: query = key = value = the raw embedding.

emb_min = embedding_matrix[TOKEN_IDS].numpy()   # (S, 3)
S_min = len(TOKENS)
QUERY = "cat"
qi = TOKENS.index(QUERY)

scores_min = emb_min @ emb_min[qi]
w_min = np.exp(scores_min - scores_min.max())
w_min /= w_min.sum()
output_min = (w_min[:, None] * emb_min).sum(0)

key_x = np.arange(S_min)
q_x = (S_min - 1) / 2.0
dim_names = ["Concrete", "Animate", "Dynamic"]

PH, REVEAL, HOLD = 4, 18, 12
plen = REVEAL + HOLD
TOTAL = PH * plen + 18


def _phase(f):
    if f >= PH * plen:
        return PH - 1, 1.0, True
    p = f // plen
    loc = f % plen
    return p, min(loc / REVEAL, 1.0), loc >= REVEAL


fig = plt.figure(figsize=(12, 7))
gs = plt.GridSpec(2, 2, height_ratios=[1.5, 1], hspace=0.5, wspace=0.25)
ax_g = fig.add_subplot(gs[0, :])
ax_w = fig.add_subplot(gs[1, 0])
ax_o = fig.add_subplot(gs[1, 1])

captions = [
    'Step 1 - pick a query token: "cat" asks "who matters to me?"',
    'Step 2 - score "cat" against every token by dot product',
    'Step 3 - softmax turns scores into weights that sum to 1',
    'Step 4 - output = weighted sum of the value vectors',
]
done_caps = [
    'Step 1 - query selected', 'Step 2 - every token scored',
    'Step 3 - weights sum to 1', 'Step 4 - context-aware vector for "cat" is ready',
]


def update(f):
    p, t, done = _phase(f)
    ax_g.clear(); ax_w.clear(); ax_o.clear()
    ax_g.set_xlim(-1, S_min); ax_g.set_ylim(-0.6, 1.7); ax_g.axis('off')
    if p >= 1:
        wshow = (scores_min - scores_min.min()) / max((scores_min.max() - scores_min.min()), 1e-9)
        reveal = t if p == 1 else 1.0
        for j in range(S_min):
            lw = 0.5 + 6 * wshow[j] * reveal
            a = min(0.15 + 0.85 * wshow[j] * reveal, 1.0)
            ax_g.plot([q_x, key_x[j]], [0, 1], color='#4c72b0', lw=lw, alpha=a, zorder=1)
    for j, tok in enumerate(TOKENS):
        ax_g.scatter(key_x[j], 1, s=520, color='#dddddd', edgecolor='#888', zorder=3)
        ax_g.text(key_x[j], 1, tok, ha='center', va='center', fontsize=9, zorder=4)
        if p >= 2:
            ax_g.text(key_x[j], 1.32, f'{w_min[j]:.2f}', ha='center', fontsize=9, color='#c44e52', fontweight='bold')
    qsize = 300 + 420 * (t if p == 0 else 1.0)
    ax_g.scatter(q_x, 0, s=qsize, color='gold', edgecolor='#b8860b', zorder=5)
    ax_g.text(q_x, 0, QUERY, ha='center', va='center', fontsize=10, fontweight='bold', zorder=6)
    ax_g.text(q_x, -0.42, 'query', ha='center', fontsize=9, color='#b8860b')
    ax_g.text((S_min-1)/2, 1.6, 'keys / values (every token)', ha='center', fontsize=9, color='#555')
    ax_w.set_xlim(-0.6, S_min-0.4); ax_w.set_ylim(0, 1.05)
    ax_w.set_xticks(key_x); ax_w.set_xticklabels(TOKENS, fontsize=8, rotation=20)
    if p == 0:
        ax_w.set_title('scores appear in step 2', fontsize=9, color='#999')
    elif p == 1:
        sc = (scores_min - scores_min.min()) / max(scores_min.max()-scores_min.min(), 1e-9)
        ax_w.bar(key_x, sc * t, color='#4c72b0', alpha=0.85)
        ax_w.set_title('Step 2 - raw dot-product scores', fontsize=10)
        ax_w.set_ylabel('score (scaled)')
    else:
        sc = (scores_min-scores_min.min()) / max(scores_min.max()-scores_min.min(), 1e-9)
        blend = (1-t)*sc + t*w_min if p == 2 else w_min
        colors = ['gold' if j==w_min.argmax() else '#4c72b0' for j in range(S_min)]
        ax_w.bar(key_x, blend, color=colors, alpha=0.85)
        ax_w.set_title('Step 3 - softmax -> weights (sum=1)' if p==2 else 'Step 3 - attention weights', fontsize=10)
        ax_w.set_ylabel('weight')
    ax_o.set_ylim(0, 1.05); ax_o.set_xticks(range(3)); ax_o.set_xticklabels(dim_names, fontsize=8)
    if p < 3:
        ax_o.bar(range(3), [0,0,0], color='#55a868'); ax_o.set_title('output builds in step 4', fontsize=9, color='#999')
    else:
        frac=t*S_min; k=int(frac); part=frac-k
        out_v=np.zeros(3)
        for j in range(min(k, S_min)): out_v += w_min[j]*emb_min[j]
        if k < S_min: out_v += part*w_min[k]*emb_min[k]
        ax_o.bar(range(3), out_v, color='#55a868')
        cur=TOKENS[min(k, S_min-1)]
        ax_o.set_title(f'Step 4 - sum w*value (adding "{cur}")' if not done else 'Step 4 - context vector', fontsize=10)
    fig.suptitle(done_caps[p] if done else captions[p], fontsize=12, fontweight='bold', color='#333')


ani = FuncAnimation(fig, update, frames=TOTAL, interval=45, blit=False, repeat=True, repeat_delay=1200)
plt.close(fig)
print('Minimal attention: query = key = value = embedding, no positions, no projections.')
top3 = w_min.argsort()[::-1][:3]
print('"cat" attends most to: ' + ", ".join(f"{TOKENS[j]} ({w_min[j]:.0%})" for j in top3))
HTML(ani.to_jshtml(default_mode='loop'))


#### What just happened - and what's missing

`"cat"` pulled most strongly toward **itself** and **`"dog"`** - its semantic neighbours. Attention found *meaning* without anyone hand-coding grammar.

But look closely at what we **never used**: *position*. Query, key and value were the raw embeddings. Shuffle the sentence and `"cat"` keeps the exact same neighbours. **Attention, on its own, is position-blind.**


---

## Part 2 - The Ordering Problem

What happens if we just **sum or average** the token vectors?

> "the cat sat on the mat"
> "mat the on sat the cat" - shuffled nonsense

Both sentences contain exactly the same words. Their mean-pooled embedding is **identical** - the model cannot tell them apart. Position information is load-bearing.


In [ ]:
#  Bag of Words - loses all positional information 
def bag_of_words(sentence: str) -> torch.Tensor:
    """Mean-pool embeddings - loses all positional information."""
    ids = torch.tensor(encode(sentence), dtype=torch.long)
    return embedding_matrix[ids].mean(dim=0)


sentences = [
    "the cat sat on the mat",
    "mat the on sat the cat",
    "sat cat mat on the the",
]
print('Mean-pooled vectors (all contain the same words):')
for s in sentences:
    v = bag_of_words(s).numpy()
    print(f"  {s!r:<42}  [{v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}]")

all_same = all(
    torch.allclose(bag_of_words(sentences[0]), bag_of_words(s))
    for s in sentences[1:]
)
print(f"\nAll three vectors identical: {all_same}")
print()
print('  -> A model with no positional encoding treats meaningful sentences and')
print('     complete nonsense as the SAME input.  We need positional encoding.')


---

## Part 3 - Positional Encoding

### 3a. Sinusoidal PE (original Transformer, "Attention Is All You Need")

$$PE_{(m,\, 2i)} = \sin\!\left(\frac{m}{10000^{2i/d}}\right)$$

$$PE_{(m,\, 2i+1)} = \cos\!\left(\frac{m}{10000^{2i/d}}\right)$$

Each dimension pair oscillates at a different frequency - a unique fingerprint for every position.


In [ ]:
#  Sinusoidal Positional Encoding 
def sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
    """Classic additive positional encoding (Vaswani et al. 2017).
    Handles both even and odd d_model gracefully.
    """
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    positions = np.arange(seq_len)[:, None].astype(np.float32)
    dims = np.arange(0, d_model, 2).astype(np.float32)
    freqs = 1.0 / (10000 ** (dims / d_model))
    pe[:, 0::2] = np.sin(positions * freqs)
    n_cos = pe[:, 1::2].shape[1]
    pe[:, 1::2] = np.cos(positions * freqs[:n_cos])
    return torch.tensor(pe)


D_VIS = 16
pe_matrix = sinusoidal_pe(SEQ_LEN, D_VIS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
im = ax.imshow(pe_matrix.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(D_VIS))
ax.set_xticklabels([f'd{i}' for i in range(D_VIS)], fontsize=8, rotation=45)
ax.set_yticks(range(SEQ_LEN)); ax.set_yticklabels(TOKENS, fontsize=10)
ax.set_title('Sinusoidal PE - our sentence')
ax.set_xlabel('Embedding dimension'); ax.set_ylabel('Token position')
plt.colorbar(im, ax=ax)

ax2 = axes[1]
pe_long = sinusoidal_pe(50, D_VIS).numpy()
for i in [0, 2, 6, 14]:
    label = f'dim {i} - {"fast" if i < 4 else "slow"}'
    ax2.plot(pe_long[:, i], label=label, lw=1.8)
ax2.set_title('PE signal per dimension over 50 positions')
ax2.set_xlabel('Token position'); ax2.set_ylabel('PE value')
ax2.legend(fontsize=8); ax2.set_ylim(-1.1, 1.1)

plt.suptitle('Sinusoidal Positional Encoding', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

emb_vectors = embedding_matrix[TOKEN_IDS]
pe_3d = sinusoidal_pe(SEQ_LEN, 3)
enriched = emb_vectors + pe_3d
print('After adding 3D sinusoidal PE:')
for i, w in enumerate(TOKENS):
    def fmt(v_list):
        return f'[{v_list[0]:+.3f}, {v_list[1]:+.3f}, {v_list[2]:+.3f}]'
    print(f'[{i}] {w:<8}  orig={fmt(emb_vectors[i].tolist())}  pe={fmt(pe_3d[i].tolist())}  sum={fmt(enriched[i].tolist())}')


#### A nagging question before we move on

Sinusoidal PE looks great in the heatmap - so why did the field move to RoPE? **Position is injected at the input, but attention projects through $W_Q$ first. If that projection smears the position signal, it was partially wasted.** RoPE fixes this by injecting position *after* the $W_Q$ projection, directly into the dot-product.


In [ ]:
#  WHY not just keep sinusoidal PE? Watch the clean signal dilute 
torch.manual_seed(0)
d_demo = 16; seq_demo = 12

pe_demo = sinusoidal_pe(seq_demo, d_demo)
content = torch.randn(seq_demo, d_demo)
W_Q_demo = torch.randn(d_demo, d_demo) * (1 / math.sqrt(d_demo))

x_in = content + pe_demo
q_proj = x_in @ W_Q_demo.T


def pos_sim(mat):
    mat = mat.detach().numpy() if isinstance(mat, torch.Tensor) else np.asarray(mat)
    m = mat / (np.linalg.norm(mat, axis=-1, keepdims=True) + 1e-9)
    return m @ m.T


sim_pe = pos_sim(pe_demo)
sim_q  = pos_sim(q_proj)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax_s, sim, title in [(axes[0], sim_pe, 'PURE sinusoidal PE\nclean diagonal band = distance-aware'),
                          (axes[1], sim_q,  'AFTER (content+PE) @ W_Q  (RANDOM W_Q)\nstructure can wash out')]:
    sns.heatmap(sim, ax=ax_s, cmap='RdBu_r', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'cosine sim'})
    ax_s.set_title(title); ax_s.set_xlabel('position'); ax_s.set_ylabel('position')
plt.suptitle('Sinusoidal PE can dilute once it is summed with content and projected', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


def monotonicity(sim):
    n = sim.shape[0]
    closeness = np.array([[-abs(i-j) for j in range(n)] for i in range(n)])
    return np.corrcoef(closeness.flatten(), sim.flatten())[0, 1]


print('"Closer positions = more similar" correlation:')
print(f'  Pure PE (at the input)         : {monotonicity(sim_pe):+.3f}   <- strong, clean')
print(f'  After content + W_Q projection : {monotonicity(sim_q):+.3f}   <- weaker')
print()
print('The principled case for RoPE: position is injected AFTER projection,')
print('so W_Q cannot dilute it, and the Q.K score depends only on (m-n) BY CONSTRUCTION.')


![Positional encoding strategies: fixed sinusoidal vs. rotary (RoPE) relative encoding](images/positional-encoding-and-rope.png)

### 3b. RoPE - Rotary Positional Embeddings

RoPE **rotates** the Query and Key vectors just before the dot-product, by an angle that depends on absolute position. The rotation cancels in a relative way - only the gap $m - n$ survives.

$$\theta_i = \frac{1}{10000^{2i/d}}$$

Token at position $m$ gets its $i$-th pair rotated by angle $m \cdot \theta_i$:

$$\text{RoPE}(x, m)_{2i:2i+2} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$


In [ ]:
#  RoPE theta values - frequency decay visualisation 
D_ROPE = 2
half = D_ROPE // 2

thetas = np.array([1.0 / (10000 ** (2 * i / D_ROPE)) for i in range(half)])
print(f'theta values for d={D_ROPE}: {thetas}')

steps = 50
angles = np.outer(np.arange(steps), thetas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for i in range(half):
    ax.plot(np.cos(angles[:, i]), label=f'Pair {i} (theta={thetas[i]:.4f})', lw=1.8)
ax.set_xlabel('Token position'); ax.set_ylabel('cos(m * theta_i)')
ax.set_title('RoPE: cosine component per dimension pair over 50 positions')
ax.legend(fontsize=8); ax.set_ylim(-1.1, 1.1)

ax2 = axes[1]
for i in range(half):
    ax2.plot(range(steps), angles[:, i] % (2 * np.pi), label=f'Pair {i}', lw=1.8)
ax2.set_xlabel('Token position'); ax2.set_ylabel('angle mod 2pi')
ax2.set_title('Accumulated rotation angle\nPair 0 spins fastest, pair 2 slowest')
ax2.legend(fontsize=8)

plt.suptitle('RoPE theta values: high-frequency pairs capture local position', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()


### 3d. Building the RoPE animation the way you'd actually discover it

Nobody arrives at a good visualisation in one shot. We build it in front of you, refinement by refinement, each step motivated by a genuine complaint about the one before.

**Step 1 - the crudest possible picture:** just plot one pair (pair 0) as a clock dial.


#### Steps 2, 3 and 4 - all at once, because the complaints compound

- **Step 2 (stacking)** - each of the 3 pairs gets its own disc at its own height.
- **Step 3 (one tower per token)** - a column per position rather than a single position.
- **Step 4 (labels)** - degree labels on each disc.


In [ ]:
#  Attempt 1 - the crudest possible RoPE picture: one dial 
th0 = 1.0 / (10000 ** (0 / 6))
circle = np.linspace(0, 2 * np.pi, 100)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), subplot_kw={'aspect': 'equal'})
for ax, m in zip(axes, range(4)):
    ang = m * th0
    ax.plot(np.cos(circle), np.sin(circle), 'lightgray', lw=1)
    ax.arrow(0, 0, math.cos(ang)*0.85, math.sin(ang)*0.85,
             head_width=0.1, head_length=0.08, fc='#4c72b0', ec='#4c72b0', lw=2)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_title(f'position {m}\nangle = {math.degrees(ang):.1f}°', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Pair 0: angle grows by theta_0 = {th0:.4f} rad per step', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()
print('Complaint: we only see one pair out of three, and only one position at a time.')


In [ ]:
#  Animated RoPE - watch each pair rotate as position increases 
RADII = [1.0, 0.7, 0.4]
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']
HEIGHTS = [1.2, 0.6, 0.0]
N_FRAMES = 60

fig3d = plt.figure(figsize=(7, 9))
ax3d = fig3d.add_subplot(111, projection='3d')


def rope_frame(step_m):
    ax3d.clear()
    circle_t = np.linspace(0, 2 * np.pi, 200)
    for pair_i, (r, col, h) in enumerate(zip(RADII, COLORS, HEIGHTS)):
        theta_i = 1.0 / (10000 ** (2 * pair_i / D_ROPE))
        ang = step_m * theta_i
        ax3d.plot(r*np.cos(circle_t), r*np.sin(circle_t), h, color=col, lw=0.5, alpha=0.3)
        ax3d.quiver(0, 0, h, r*math.cos(ang), r*math.sin(ang), 0, color=col, lw=2, arrow_length_ratio=0.15)
        ax3d.text(r*1.05, 0, h, f'Pair {pair_i}', fontsize=8, color=col)
    ax3d.set_xlim(-1.3, 1.3); ax3d.set_ylim(-1.3, 1.3); ax3d.set_zlim(-0.3, 1.7)
    ax3d.set_xlabel('cos', fontsize=7); ax3d.set_ylabel('sin', fontsize=7)
    ax3d.set_title(f'RoPE rotation at position m={step_m}', fontsize=10)
    ax3d.tick_params(labelsize=7)


ani3d = FuncAnimation(fig3d, rope_frame, frames=N_FRAMES, interval=80, repeat=True)
plt.close(fig3d)
print('Each disc = one dimension pair. Arrow angle = m * theta_i.')
print('Pair 0 (blue) spins fastest; pair 2 (green) barely moves.')
HTML(ani3d.to_jshtml(default_mode='loop'))


In [ ]:
#  RoPE implementation + relative-distance proof 

def rope_rotate(x, m: int, thetas_arr: np.ndarray) -> np.ndarray:
    """Rotate vector x (shape: d) at token position m using RoPE."""
    x_rot = np.array(x, dtype=np.float32).copy()
    for i, th in enumerate(thetas_arr):
        angle = m * th
        c, s = math.cos(angle), math.sin(angle)
        a, b = x_rot[2*i], x_rot[2*i+1]
        x_rot[2*i]   = a*c - b*s
        x_rot[2*i+1] = a*s + b*c
    return x_rot


thetas_demo = np.array([1.0 / (10000 ** (2*i/D_ROPE)) for i in range(D_ROPE//2)])
cat_emb = embedding_matrix[VOCAB['cat']].numpy()[:D_ROPE]
mat_emb = embedding_matrix[VOCAB['mat']].numpy()[:D_ROPE]

print('Relative-distance property of RoPE:')
print('  cat at pos m, mat at pos n: Q_cat * K_mat depends only on (m-n)')
print()

for gap in [1, 2, 3]:
    results = []
    for base in [0, 1, 2, 3]:
        q = rope_rotate(cat_emb, base, thetas_demo)
        k = rope_rotate(mat_emb, base + gap, thetas_demo)
        results.append(float(q @ k))
    print(f'  gap={gap}: dot products = {[f"{r:.4f}" for r in results]}  '
          f'(all equal -> {np.allclose(results, results[0], atol=1e-5)})')

print()
print('  -> RoPE guarantees relative position: the dot product depends ONLY on (m-n).')


---

## Part 4 - Queries, Keys & Values

The transformer's attention mechanism is a **soft dictionary lookup**.

| Component | Intuition | Created by |
| --------- | --------- | ---------- |
| **Q** (Query) | "What am I looking for?" | $W_Q \cdot x$ |
| **K** (Key)   | "What do I advertise?"   | $W_K \cdot x$ |
| **V** (Value) | "What do I contribute?"  | $W_V \cdot x$ |

$W_Q$, $W_K$, $W_V$ are **learned** projection matrices. Each head has its own set.


![Q, K, V data flow: query selects, key gates, value delivers](images/attention-qkv-data-flow.png)

In [ ]:
#  Q / K / V projections in 3D space 
torch.manual_seed(7)
D_MODEL = 3

W_Q = torch.randn(D_MODEL, D_MODEL) * 0.5
W_K = torch.randn(D_MODEL, D_MODEL) * 0.5
W_V = torch.randn(D_MODEL, D_MODEL) * 0.5

embs = embedding_matrix[TOKEN_IDS]   # (6, 3)

Q = embs @ W_Q.T   # (6, 3)
K = embs @ W_K.T
V = embs @ W_V.T

fig = plt.figure(figsize=(15, 4))
titles = ['Input Embeddings', 'Queries  (W_Q @ x)', 'Keys  (W_K @ x)', 'Values  (W_V @ x)']
arrays = [embs, Q, K, V]
colors = plt.cm.tab10(np.linspace(0, 0.6, SEQ_LEN))

for idx, (title, arr) in enumerate(zip(titles, arrays)):
    ax = fig.add_subplot(1, 4, idx + 1, projection='3d')
    arr_np = arr.detach().numpy()
    for j, (word, vec) in enumerate(zip(TOKENS, arr_np)):
        ax.scatter(*vec, color=colors[j], s=70, zorder=5)
        ax.text(*vec, f' {word}', fontsize=7, color=colors[j])
    ax.set_title(title, fontsize=9, pad=6)
    ax.set_xlabel('d0', fontsize=7); ax.set_ylabel('d1', fontsize=7); ax.set_zlabel('d2', fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle('How W_Q, W_K, W_V rotate/stretch the embedding space', fontsize=11, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
#  Scaled Dot-Product Attention 
def scaled_attention(Q_in: torch.Tensor, K_in: torch.Tensor, V_in: torch.Tensor, mask=None):
    """Scaled dot-product attention. Returns (output, attn_weights).
    Q_in, K_in, V_in: (seq_len, d_k)
    """
    d_k = Q_in.shape[-1]

    scores = Q_in @ K_in.T / math.sqrt(d_k)
    print(f'  Raw score matrix (QK^T / sqrt(d_k)):\n  {scores.detach().numpy().round(3)}')

    if mask is not None:
        scores = scores.masked_fill(mask.bool(), float('-inf'))

    attn_w = torch.softmax(scores, dim=-1)
    out = attn_w @ V_in
    return out, attn_w


print('=== Step-by-step attention on our sentence ===')
print(f'Q shape: {tuple(Q.shape)}  K shape: {tuple(K.shape)}  V shape: {tuple(V.shape)}\n')

out, attn_w = scaled_attention(Q, K, V)

print(f'\n  Attention weight matrix (row = query token, col = key token):')
print(f'  Tokens: {TOKENS}')
print(f'  {attn_w.detach().numpy().round(3)}')
print(f'\n  Output shape: {tuple(out.shape)}')


In [ ]:
#  Attention heatmap visualisation 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
w = attn_w.detach().numpy()
sns.heatmap(w, ax=ax, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
            cbar_kws={'label': 'attention weight'})
ax.set_title('Bidirectional Attention (encoder-style)')
ax.set_xlabel('Key token'); ax.set_ylabel('Query token'); ax.tick_params(axis='x', rotation=30)

ax2 = axes[1]
S6 = SEQ_LEN
causal_mask_vis = torch.triu(torch.ones(S6, S6, dtype=torch.bool), diagonal=1)
_, attn_w_causal = scaled_attention(Q, K, V, mask=causal_mask_vis)
wc = attn_w_causal.detach().numpy()
sns.heatmap(wc, ax=ax2, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
            cbar_kws={'label': 'attention weight'})
ax2.set_title('Causal Attention (decoder-style)\nupper triangle masked to -inf')
ax2.set_xlabel('Key token'); ax2.set_ylabel('Query token'); ax2.tick_params(axis='x', rotation=30)

plt.suptitle('Bidirectional vs Causal Attention', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


### Your turn - attention

You've watched attention; now drive it. Change `my_query` and **predict the top attention target before you run it**.


In [ ]:
#  EXERCISE 1 - attention by hand 
# Change `my_query` to any token in TOKENS and PREDICT its top attention target BEFORE running.
# TOKENS = ['the', 'cat', 'sat', 'on', 'the', 'mat']
my_query = 'cat'   # try 'sat', 'mat', 'on', ...

qi = TOKENS.index(my_query)
scores_ex = Q @ K.T / math.sqrt(Q.shape[-1])
w_ex = torch.softmax(scores_ex, dim=-1).detach().numpy()[qi]

print(f'"{my_query}" (position {qi}) attends most to:')
for r in w_ex.argsort()[::-1][:3]:
    print(f'   {TOKENS[r]:<8} (pos {r})  weight={w_ex[r]:.3f}')


In [ ]:
#  RoPE applied to Q and K inside attention - step 1: rotate Q and K 
def apply_rope_to_qk(Q_in, K_in, thetas_arr: np.ndarray) -> tuple:
    """Apply RoPE to Q and K (seq_len x d). Returns (Q_rot, K_rot) as torch.Tensor."""
    Q_np = Q_in.detach().numpy() if isinstance(Q_in, torch.Tensor) else np.asarray(Q_in)
    K_np = K_in.detach().numpy() if isinstance(K_in, torch.Tensor) else np.asarray(K_in)
    seq_len, d = Q_np.shape
    n_pairs = d // 2
    positions = np.arange(seq_len, dtype=np.float32)

    def rope(x):
        out_r = x.copy()
        for i in range(n_pairs):
            ang = positions * float(thetas_arr[i])
            cos_a, sin_a = np.cos(ang), np.sin(ang)
            a, b = x[:, 2*i], x[:, 2*i+1]
            out_r[:, 2*i]   = a*cos_a - b*sin_a
            out_r[:, 2*i+1] = a*sin_a + b*cos_a
        return out_r

    return torch.tensor(rope(Q_np)), torch.tensor(rope(K_np))


th_vis = np.array([1.0 / (10000 ** (2*i/D_MODEL)) for i in range(D_MODEL // 2)])
Q_raw, K_raw = Q, K
Q_rot, K_rot = apply_rope_to_qk(Q_raw, K_raw, th_vis)

print(f'{"Token":<8}  {"Q_raw":>26}  {"Q_rotated (RoPE)":>26}  {"delta norm":>10}')
print('  ' + '-' * 74)
for i, tok in enumerate(TOKENS):
    qr, qn = Q_raw[i].numpy(), Q_rot[i].numpy()
    delta = np.linalg.norm(qn - qr)
    print(f'  {tok:<8}  [{qr[0]:+.3f}, {qr[1]:+.3f}, {qr[2]:+.3f}]  '
          f'[{qn[0]:+.3f}, {qn[1]:+.3f}, {qn[2]:+.3f}]  {delta:>10.4f}')
print()
print("Notice: 'the' at position 0 has m=0, so angle=0 -> Q_rotated = Q_raw.")


#### Does that rotation actually change what attends to what?

Same projections, one difference - RoPE applied or not - side by side.


In [ ]:
#  RoPE in attention - step 2: the payoff on attention weights 
scores_raw = Q_raw @ K_raw.T / math.sqrt(D_MODEL)
scores_rot = Q_rot @ K_rot.T / math.sqrt(D_MODEL)
attn_raw = torch.softmax(scores_raw, dim=-1)
attn_rot = torch.softmax(scores_rot, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, w, title in zip(axes, [attn_raw, attn_rot], ['Attention WITHOUT RoPE', 'Attention WITH RoPE']):
    sns.heatmap(w.detach().numpy(), ax=ax, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('RoPE shifts attention weights by baking position into Q*K scores', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


### 4c. Discovering the attention formula - softmax and sqrt(d)

#### Predict first

1. Raw $QK^T$ scores can be negative and don't sum to 1. **What single operation turns an arbitrary score vector into a probability distribution?**
2. In a real model $d_k$ is 64-128. The dot product of two random vectors grows with dimension. **What happens to softmax when scores get very large?**


In [ ]:
#  DISCOVERING softmax and the sqrt(d) scale - decision 1: why softmax? 
raw = (Q_rot @ K_rot.T)[0].detach().numpy()
print('Raw QK^T scores for one query row:')
print('  ', raw.round(3))
print(f'  sum = {raw.sum():+.3f}  (not 1)  and some are negative -> NOT a probability.')
print('  softmax fixes both: exp() makes them positive, then normalise to sum = 1.')
print()
print('  -> Softmax is the only differentiable function that produces a probability')
print('     distribution from arbitrary real-valued scores. -> conclusion')


#### Decision 2 - why divide by sqrt(d)?

Softmax alone isn't enough. The dot product of two random vectors has variance that **grows with dimension**.


In [ ]:
#  Decision 2, step 1: dot-product variance grows with dimension 
dims = [4, 16, 64, 256, 1024]
print(f'{"d_k":>6} | {"std(QK^T) unscaled":>18} | {"std after /sqrt(d_k)":>15}')
print('  ' + '-' * 46)
peak_unscaled, peak_scaled = [], []
for d in dims:
    q_v = torch.randn(4000, d)
    k_v = torch.randn(4000, d)
    dot = (q_v * k_v).sum(dim=-1).numpy()
    print(f'{d:>6} | {dot.std():>18.2f} | {(dot / math.sqrt(d)).std():>15.2f}')
    qr2 = torch.randn(500, 8, d)
    kr2 = torch.randn(500, 8, d)
    s_un = (qr2 * kr2).sum(dim=-1)
    s_sc = s_un / math.sqrt(d)
    peak_unscaled.append(float(torch.softmax(s_un, dim=-1).max(dim=-1).values.mean()))
    peak_scaled.append(float(torch.softmax(s_sc, dim=-1).max(dim=-1).values.mean()))
print('  Unscaled variance grows like sqrt(d); dividing by sqrt(d) pins it ~1 at every width.')


#### The consequence - saturation kills the gradient

Big scores push softmax toward a one-hot spike. A one-hot softmax has almost no slope, so the gradient vanishes.


In [ ]:
#  Decision 2, step 2: the CONSEQUENCE - saturation kills the gradient 
d_big = 256
k_sat = torch.randn(8, d_big)
for label, scale in [('WITHOUT /sqrt(d)', 1.0), ('WITH /sqrt(d)', math.sqrt(d_big))]:
    q_sat = torch.randn(8, d_big, requires_grad=True)
    p = torch.softmax((q_sat @ k_sat.T) / scale, dim=-1)
    loss_sat = torch.sum(torch.max(p, dim=-1).values)
    loss_sat.backward()
    g = q_sat.grad
    peak_p = torch.max(p, dim=-1).values.mean().item()
    print(f'  {label:<16}: mean peak prob = {peak_p:.3f}   |grad(q)| = {torch.norm(g).item():.2e}')

print()
print('WITHOUT scaling: softmax -> one-hot (peak~1) -> gradient ~0 -> layer cannot learn.')
print('WITH   scaling: distribution stays soft -> gradient flows -> training works.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims, peak_unscaled, 'o-', color='tomato',   lw=2, label='unscaled QK^T')
ax.plot(dims, peak_scaled,   'o-', color='seagreen', lw=2, label='scaled QK^T/sqrt(d)')
ax.axhline(1/8, color='gray', ls='--', lw=1, label='uniform (1/8)')
ax.set_xscale('log', base=2); ax.set_xlabel('d_k  (attention head dimension)')
ax.set_ylabel('mean peak softmax probability')
ax.set_title('Without sqrt(d) scaling, softmax saturates to one-hot as d_k grows')
ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout(); plt.show()


---

## Part 5 - Multi-Head Attention

**Multi-Head Attention** (MHA) runs $H$ parallel attention heads:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \cdot W_O$$

**Why multiple heads?** Each head can specialise on a different relationship type.


In [ ]:
#  Working model constants 
D_WORK = 16    # functional model dimension
NUM_HEADS = 2  # attention heads
D_HEAD = D_WORK // NUM_HEADS   # 8 per head
D_FF = 32      # feed-forward hidden size


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, _ = x.shape
        Q_mh = self.W_Q(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, S, d_head)
        K_mh = self.W_K(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V_mh = self.W_V(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q_mh @ K_mh.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, S, S)
        if mask is not None:
            scores = scores.masked_fill(mask.bool(), float('-inf'))
        attn_w_mh = torch.softmax(scores, dim=-1)   # (B, H, S, S)
        out = attn_w_mh @ V_mh                       # (B, H, S, d_head)
        out = out.transpose(1, 2).reshape(B, S, self.d_model)
        return self.W_O(out), attn_w_mh


#  Demo 
torch.manual_seed(42)
mha = MultiHeadAttention(D_WORK, NUM_HEADS)

proj = nn.Linear(D_MODEL, D_WORK, bias=False)
with torch.no_grad():
    x_work = proj(embs).unsqueeze(0)   # (1, 6, 16)

mha_out, head_weights = mha(x_work)
print(f'MHA output shape: {tuple(mha_out.shape)}   head_weights shape: {tuple(head_weights.shape)}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in range(NUM_HEADS):
    ax = axes[h]
    w_h = head_weights[0, h].detach().numpy()
    sns.heatmap(w_h, ax=ax, annot=True, fmt='.2f', cmap='Purples',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(f'Head {h} attention weights')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Multi-Head Attention - each head learns a different relationship', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


#### Predict first - can one head do two jobs?

A single attention head produces **one** score matrix. Suppose you want every token to attend to **both** its previous token and its most semantically similar token.

**Predict:** can one head satisfy both relations at once, or must it pick one?


In [ ]:
#  PROVING the multi-head claim - step 1: two heads, two patterns 
torch.manual_seed(0)
S = SEQ_LEN
X = embs   # (S, 3)
V_shared = X @ torch.randn(3, 3)

# Relation P (positional): attend to the PREVIOUS token
prev_idx = np.array([max(i-1, 0) for i in range(S)])
scores_pos = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S):
    scores_pos[i, prev_idx[i]] = 9.0
A_pos = torch.softmax(torch.tensor(scores_pos), dim=-1)

# Relation C (content): attend to the most SEMANTICALLY SIMILAR token
sim = (X @ X.T).detach().numpy()
np.fill_diagonal(sim, -1e9)
near_idx = sim.argmax(-1)
scores_con = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S):
    scores_con[i, near_idx[i]] = 9.0
A_con = torch.softmax(torch.tensor(scores_con), dim=-1)

corr = np.corrcoef(A_pos.numpy().flatten(), A_con.numpy().flatten())[0, 1]
print(f'Correlation between the two head patterns: {corr:+.3f}   (~0 -> different information)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, A, title, cmap in [
    (axes[0], A_pos.numpy(), 'Head P - positional (attend to previous token)', 'Greens'),
    (axes[1], A_con.numpy(), 'Head C - content (attend to most similar token)', 'Purples'),
]:
    sns.heatmap(A, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=10); ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Two heads, two DIFFERENT relations', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


#### So they differ - but could a *single* head carry both?

Each head yields exactly **one** output per token. Let's measure how well each head recovers each relation.


In [ ]:
#  PROVING the multi-head claim - step 2: one head can't do both 
target_prev = V_shared[prev_idx]
target_near = V_shared[near_idx]
out_pos = A_pos @ V_shared
out_con = A_con @ V_shared


def mse(a, b):
    return float(((a - b) ** 2).mean())


print('Reconstruction error (lower = that relation is captured):')
print(f'  Head P alone -> previous-token target : {mse(out_pos, target_prev):.4f}   <- nails it')
print(f'  Head P alone -> similar-token  target : {mse(out_pos, target_near):.4f}   <- misses it')
print(f'  Head C alone -> previous-token target : {mse(out_con, target_prev):.4f}   <- misses it')
print(f'  Head C alone -> similar-token  target : {mse(out_con, target_near):.4f}   <- nails it')
print()
print('  -> A single head serves ONE relation well, never both.')
print('  -> Concatenating [Head P ; Head C] delivers BOTH targets in parallel.')
print('  -> That is why H heads exist: H independent relations, computed at once.')


### Your turn - heads

Dial the number of heads up and down and watch how independent their patterns become.


In [ ]:
#  EXERCISE 2 - how many heads? 
# Change `n_heads` (must divide D_WORK = 16: try 1, 2, 4, 8).
# Predict: more heads = more independent relations captured at once.
n_heads = 1   # try 1, then 4, then 8

torch.manual_seed(42)
mha_ex = MultiHeadAttention(D_WORK, n_heads)
_, w_ex = mha_ex(x_work)
print(f'{n_heads} head(s) -> {n_heads} attention pattern(s), each {D_WORK // n_heads}-dim wide.')

pats = w_ex[0].detach().reshape(n_heads, -1).numpy()
if n_heads > 1:
    corrs = np.corrcoef(pats)
    print('Pairwise correlations between head patterns:')
    for hi in range(n_heads):
        for hj in range(hi + 1, n_heads):
            print(f'  head {hi} vs head {hj}: r = {corrs[hi, hj]:+.3f}')
else:
    print('  (only one head - no pairwise comparison)')


---

## Part 6 - Feed-Forward Network & Layer Normalisation

### Feed-Forward Network (FFN)

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1)\, W_2 + b_2$$

Expands by 4x then projects back. Adds non-linear transformation capacity.

### Layer Normalisation

Applied **before** each sub-layer (Pre-LN style). Normalises each token's vector to zero mean and unit variance.


In [ ]:
#  FeedForward + LayerNorm 
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


#  Visualise LayerNorm effect 
torch.manual_seed(42)
ffn = FeedForward(D_WORK, D_FF)
norm = nn.LayerNorm(D_WORK, eps=1e-5)

x_raw = x_work[0]   # (6, 16)
with torch.no_grad():
    x_after = ffn(x_raw)       # (6, 16) raw FFN output
    x_normed = norm(x_after)   # (6, 16) after LayerNorm

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [x_raw, x_after, x_normed],
                           ['Input to FFN', 'FFN output (raw)', 'After LayerNorm']):
    data_np = data.detach().numpy()
    for j, token in enumerate(TOKENS):
        vals = data_np[j]
        ax.plot(vals, alpha=0.7, label=f'{token}  mu={vals.mean():.2f}, s={vals.std():.2f}')
    ax.set_title(title); ax.set_xlabel('Hidden dimension'); ax.set_ylabel('Activation value')
    ax.legend(fontsize=7); ax.axhline(0, color='black', lw=0.5, ls='--')
plt.suptitle('FFN activations before and after LayerNorm', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('LayerNorm centres and normalises each token slice.')
print('Mean and std across dimensions after LN:')
x_normed_np = x_normed.detach().numpy()
for j, token in enumerate(TOKENS):
    v = x_normed_np[j]
    print(f'  {token:<8}  mean={v.mean():+.4f}  std={v.std():.4f}')


![Complete Transformer block: LayerNorm → Multi-Head Attention → Residual → LayerNorm → FFN → Residual](images/transformer-block-overview.png)

---

## Part 7 - Full Transformer Block & RNN Comparison

A single **Transformer Block** wires MHA + FFN together with layer norm and residuals:

```
x --> LayerNorm --> MHA --> (+x) --> LayerNorm --> FFN --> (+x) --> output
```

Stack $L$ of these blocks = the full encoder/decoder stack.


In [ ]:
#  TransformerBlock 
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.mha   = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn   = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        mha_out, attn_w_b = self.mha(self.norm1(x), mask=mask)
        x = x + mha_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_w_b


torch.manual_seed(42)
block1 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)
block2 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)

x0 = x_work.clone()   # (1, 6, 16)
with torch.no_grad():
    x1, aw1 = block1(x0)
    x2, aw2 = block2(x1)

print('Input -> Block 1 -> Block 2:')
print(f'  x0: {tuple(x0.shape)}  norm={float(torch.norm(x0)):.3f}')
print(f'  x1: {tuple(x1.shape)}  norm={float(torch.norm(x1)):.3f}')
print(f'  x2: {tuple(x2.shape)}  norm={float(torch.norm(x2)):.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
norms = {
    'Layer 0 (input)': torch.norm(x0[0], dim=-1).detach().numpy(),
    'Layer 1 output':  torch.norm(x1[0], dim=-1).detach().numpy(),
    'Layer 2 output':  torch.norm(x2[0], dim=-1).detach().numpy(),
}
x_pos = np.arange(SEQ_LEN); width = 0.25
for k, (label, vals) in enumerate(norms.items()):
    ax.bar(x_pos + k*width, vals, width, label=label, alpha=0.85)
ax.set_xticks(x_pos + width); ax.set_xticklabels(TOKENS)
ax.set_ylabel('Representation L2 norm')
ax.set_title('Token representations grow through transformer blocks')
ax.legend(); plt.tight_layout(); plt.show()


#### Predict first - does the skip connection really matter?

We'll stack **24** simple layers and read the gradient that reaches **layer 1** (furthest from the loss), with and without the skip $x + \text{SubLayer}(x)$.

**Predict:** without the skip, will the gradient at layer 1 be vanishingly small?


In [ ]:
#  DEMONSTRATING why residual connections make depth trainable 

DEPTH = 24
d = D_WORK


class ProbeBlock(nn.Module):
    def __init__(self, d_in, residual):
        super().__init__()
        self.lin = nn.Linear(d_in, d_in)
        self.residual = residual

    def forward(self, x):
        y = torch.tanh(self.lin(x))
        return x + y if self.residual else y


def gradient_reaching_each_layer(residual):
    torch.manual_seed(0)
    blocks = nn.ModuleList([ProbeBlock(d, residual) for _ in range(DEPTH)])
    x_p = torch.randn(1, d)
    h = x_p
    for b in blocks:
        h = b(h)
    loss_p = torch.mean(h ** 2)
    loss_p.backward()
    return [b.lin.weight.grad.norm().item() for b in blocks]


g_res   = gradient_reaching_each_layer(residual=True)
g_plain = gradient_reaching_each_layer(residual=False)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(range(1, DEPTH+1), g_plain, 'o-', color='tomato',   lw=2, label='WITHOUT residual')
ax.plot(range(1, DEPTH+1), g_res,   'o-', color='seagreen', lw=2, label='WITH residual')
ax.set_yscale('log')
ax.set_xlabel('Layer (1 = furthest from loss, closest to input)')
ax.set_ylabel('|gradient| reaching this layer  (log scale)')
ax.set_title('Residual connections keep gradients alive all the way to layer 1')
ax.legend(); plt.tight_layout(); plt.show()

print('Gradient norm reaching layer 1 (the earliest, hardest-to-train layer):')
print(f'  WITHOUT residual: {g_plain[0]:.2e}   <- vanished')
print(f'  WITH    residual: {g_res[0]:.2e}   <- healthy')
ratio = g_res[0] / max(g_plain[0], 1e-30)
print(f'  The skip path delivers ~{ratio:.1e}x more gradient to the earliest layer.')


### Your turn - depth

Push the stack deeper and watch the no-residual gradient collapse while the residual one stays alive.


In [ ]:
#  EXERCISE 3 - how deep can you go WITHOUT residuals? 
# Change `DEPTH` (try 8, 24, 60) and PREDICT how far the gradient survives.
DEPTH = 40

g_res_ex   = gradient_reaching_each_layer(residual=True)
g_plain_ex = gradient_reaching_each_layer(residual=False)
print(f'At depth {DEPTH}, gradient reaching layer 1 (the hardest to train):')
print(f'  WITHOUT residual: {g_plain_ex[0]:.2e}')
print(f'  WITH    residual: {g_res_ex[0]:.2e}')
ratio_ex = g_res_ex[0] / max(g_plain_ex[0], 1e-30)
print(f'  -> Ratio: {ratio_ex:.1e}x more gradient with residuals.')


---

## Part 8 - Mini Language Model: Training & Inference

We now wire everything together into a **Mini Language Model** - a decoder-only transformer that learns to predict the next token. This is the architecture of GPT, LLaMA, Mistral etc.

**Training task**: given a context window, predict the next token.


In [ ]:
#  MiniLM - decoder-only transformer language model 
class MiniLM(nn.Module):
    """
    Decoder-only transformer language model.
    Architecture: token_emb -> sinusoidal_PE -> n_layers x TransformerBlock -> lm_head
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model, eps=1e-5)
        pe = sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)

    def forward(self, token_ids, return_attn=False):
        """
        token_ids: (batch, seq_len)  int tensor
        Returns logits: (batch, seq_len, vocab_size)
        """
        S = token_ids.shape[1]
        x = self.token_emb(token_ids) + self.pe[:S]   # (B, S, d_model)
        causal_mask = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1).to(x.device)
        all_attn = []
        for block in self.blocks:
            x, aw = block(x, mask=causal_mask)
            all_attn.append(aw)
        x = self.norm_out(x)
        # Weight tying: reuse token embedding matrix as output projection
        logits = x @ self.token_emb.weight.T
        if return_attn:
            return logits, all_attn
        return logits


torch.manual_seed(42)
model_demo = MiniLM(vocab_size=VOCAB_SIZE, d_model=D_WORK, n_heads=NUM_HEADS, d_ff=D_FF, n_layers=2)
with torch.no_grad():
    _ = model_demo(torch.tensor([TOKEN_IDS]))
n_params = sum(p.numel() for p in model_demo.parameters())
print(f'MiniLM -> {n_params:,} trainable parameters')
for name, p in model_demo.named_parameters():
    print(f'  {name:<40} {tuple(p.shape)}')


In [ ]:
#  Training data - (context, next_token) pairs 
full_corpus = [
    "the cat sat on the mat",
    "the dog ran over the fence",
    "a big cat jumped over the fence",
    "a dog sat on the mat",
    "the cat jumped over the fence",
    "the big dog ran on the mat",
]

TRAIN_PAIRS = []
for sentence in full_corpus:
    ids = encode(sentence)
    for end in range(1, len(ids)):
        TRAIN_PAIRS.append((ids[:end], ids[end]))

print(f'Training pairs: {len(TRAIN_PAIRS)}')
print('\nFirst 6 examples:')
for ctx, tgt in TRAIN_PAIRS[:6]:
    print(f'  {[IDX2WORD[i] for i in ctx]}  ->  "{IDX2WORD[tgt]}"')


Now the training loop: full-batch gradient descent, predicting each next token from the position before it.


In [ ]:
#  Training loop 
def pad_collate(pairs, pad_id=0):
    max_len = max(len(ctx) for ctx, _ in pairs)
    xs, ys = [], []
    for ctx, tgt in pairs:
        pad = [pad_id] * (max_len - len(ctx))
        xs.append(pad + ctx)
        ys.append(tgt)
    return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)


torch.manual_seed(42)
model = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 300
x_train, y_train = pad_collate(TRAIN_PAIRS)
loss_history, acc_history = [], []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    last_logits = logits[:, -1, :]
    loss = loss_fn(last_logits, y_train)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            preds = torch.argmax(last_logits, dim=-1)
            acc = (preds == y_train).float().mean().item()
        loss_history.append(loss.item())
        acc_history.append(acc)
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:4d} | loss={loss.item():.4f} | acc={acc:.2%}')

print('\nTraining complete.')


In [ ]:
#  Training loss & accuracy plot 
epochs_logged = list(range(10, EPOCHS + 1, 10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(epochs_logged, loss_history, color='royalblue', lw=2)
ax.fill_between(epochs_logged, loss_history, alpha=0.15, color='royalblue')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss'); ax.set_title('Training Loss')

ax2 = axes[1]
ax2.plot(epochs_logged, [a * 100 for a in acc_history], color='mediumseagreen', lw=2)
ax2.fill_between(epochs_logged, [a * 100 for a in acc_history], alpha=0.15, color='mediumseagreen')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Next-Token Prediction Accuracy')
ax2.set_ylim(0, 105); ax2.axhline(100, color='grey', ls='--', lw=0.8)

plt.suptitle('MiniLM Training Progress', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


![Autoregressive generation and KV cache: how the model produces one token at a time](images/autoregressive-generation-and-kv-cache.png)

---

## Part 9 - Inference: Autoregressive Token Generation

At inference time a language model generates text **one token at a time**:

1. Feed the current context into the model
2. Take the logits at the **last position**
3. Apply temperature scaling + softmax
4. Sample (or argmax = greedy)
5. Append the new token -> go to step 1


In [ ]:
#  Inference - autoregressive token generation 
def generate_next(context_words, temperature=1.0):
    """Single next-token generation step with probability bar chart."""
    model.eval()
    with torch.no_grad():
        ids = torch.tensor([encode(' '.join(context_words))], dtype=torch.long)
        logits_inf = model(ids)
        last = logits_inf[0, -1, :]
        probs = torch.softmax(last / max(temperature, 1e-6), dim=-1).numpy()

    topk_idx = probs.argsort()[::-1][:8]
    top_words = [IDX2WORD[int(i)] for i in topk_idx]
    top_probs = probs[topk_idx]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(top_words[::-1], top_probs[::-1],
            color=['gold' if w == top_words[0] else 'steelblue' for w in top_words[::-1]])
    ax.set_xlabel('Probability')
    ax.set_title(f'Next token probabilities | context: {context_words}  T={temperature}')
    ax.set_xlim(0, 1.0)
    for bar, prob in zip(ax.patches, top_probs[::-1]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9)
    plt.tight_layout(); plt.show()

    best = IDX2WORD[int(probs.argmax())]
    print(f'  Greedy prediction: "{best}"')
    return best


#  Demo: step-by-step generation 
print('=== Autoregressive generation ===')
print()
context = ['the']
for step in range(5):
    print(f'Step {step+1}: context = {context}')
    next_tok = generate_next(context, temperature=0.8)
    context.append(next_tok)
    print()

print(f'Generated sequence: {" ".join(context)}')


![Scaling from toy (d=3) to production (d=768): same architecture, different dimensions](images/toy-to-production-transformers.png)

### From toy to real - same mechanism, bigger numbers

Everything you've built used tiny dimensions so the vectors stayed readable. A production model is the **identical machinery** scaled up.


In [ ]:
#  Toy (this notebook) vs. a real model (GPT-2 / DistilGPT-2) 
rows = [
    ('embedding dim  d_model', D_MODEL, D_WORK, 768),
    ('attention heads',        '?',     NUM_HEADS, 12),
    ('dim per head  d_head',   '?',     D_HEAD, 64),
    ('feed-forward hidden',    '?',     D_FF, 3072),
    ('transformer layers',     '?',     2, 12),
    ('vocabulary size',        VOCAB_SIZE, VOCAB_SIZE, 50257),
]
print(f'{"component":<24}{"viz":>8}{"toy model":>12}{"GPT-2":>10}')
print('  ' + '-' * 52)
for name, viz, toy, real in rows:
    print(f'  {name:<22} {str(viz):>8} {str(toy):>12} {str(real):>10}')
print()
print('Every component in GPT-2 is identical in kind to what you built.')
print('  -> Scale, not novelty, is what makes GPT-2 impressive.')


---

## Part 11 - Why W_V Is a Relevance Filter, Not a Passthrough

$W_V$ is a **task-specific extraction lens**. Each attention layer has a different job. $W_V$ lets each layer extract exactly the slice it needs from each token's information.


#### Predict first - do we even need W_V?

Suppose we deleted $W_V$ entirely and blended the **raw token embeddings** directly.

**Predict:** if a *grammar* attention layer wants to know "is this token part of an active event?", it cares mainly about Dynamism and Animacy. Without $W_V$, can it ignore Concreteness?


In [ ]:
#  Part 11: W_V as a relevance filter 
embs_sentence = embedding_matrix[TOKEN_IDS].numpy()   # (6, 3)

# W_V_action: extract the ACTION channel (Animacy + Dynamism)
W_V_action = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], dtype=np.float32)

# W_V_object: extract the OBJECT channel (Concreteness + Animacy)
W_V_object = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]], dtype=np.float32)

V_action = embs_sentence @ W_V_action   # (6, 2)
V_object = embs_sentence @ W_V_object   # (6, 2)

attn_row_cat = attn_w.detach().numpy()[TOKENS.index('cat')]
blend_action = (attn_row_cat[:, None] * V_action).sum(0)
blend_object = (attn_row_cat[:, None] * V_object).sum(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
specs = [
    (axes[0], V_action, blend_action, 'W_V_action - ACTION lens\n(Animacy x Dynamism)', 'Animacy', 'Dynamism'),
    (axes[1], V_object, blend_object, 'W_V_object - OBJECT lens\n(Concreteness x Animacy)', 'Concreteness', 'Animacy'),
]
for ax, V, blend, title, xl, yl in specs:
    ax.scatter(V[:, 0], V[:, 1], s=80, c='lightgray', edgecolor='#888', zorder=3)
    for i, tok in enumerate(TOKENS):
        ax.annotate(f'[{i}]{tok}', (V[i, 0], V[i, 1] + 0.03), fontsize=8, ha='center', color='dimgray')
    ax.scatter(*blend, s=280, color='gold', edgecolor='#b8860b', zorder=5,
               label='"cat" blended context', linewidths=2)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title, fontsize=10); ax.legend(fontsize=9)
plt.suptitle('W_V shapes WHAT part of each token enters the weighted blend.\n'
             'Same sentence, same attention weights - different W_V - different context vector.',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('Key insight:')
print('  Attention weights (W_Q/W_K path) answer WHO gets blended.')
print('  W_V answers WHAT each token contributes to that blend.')
print('  -> W_V is not a passthrough; it is a task-specific extraction lens.')


---

## Part 12 - The Causal Triangle and the Accumulation Tower

The causal mask determines *how much of the sentence each position gets to know about*. Position 0 sees only itself. Position 5 sees all six tokens.

> **By the time the last position exits the final transformer block, it has absorbed a chain of increasingly enriched representations from every earlier position.**


In [ ]:
#  The Causal Triangle - explicit lower-triangular structure 
S = SEQ_LEN
mask_vis = np.tril(np.ones((S, S), dtype=int))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
sns.heatmap(mask_vis.astype(float), ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=1.0, cbar=False,
            annot=mask_vis, fmt='d', annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Causal Mask\n1 = allowed to attend  |  0 = blocked')
ax.set_xlabel('Key token'); ax.set_ylabel('Query token')

ax2 = axes[1]
history_counts = np.arange(1, S + 1)
bar_colors = plt.cm.Blues(np.linspace(0.35, 0.9, S))
ax2.bar(range(S), history_counts, color=bar_colors, edgecolor='white', lw=1)
ax2.set_xticks(range(S)); ax2.set_xticklabels(TOKENS)
ax2.set_ylabel('Tokens visible to this position')
ax2.set_title('How many tokens each position knows about')
for i, c in enumerate(history_counts):
    ax2.text(i, c + 0.05, str(c), ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Causal Triangle: position 0 is isolated; position 5 absorbs all 6 tokens',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


#### Predict first - does the accumulated history actually matter?

- **Full** context: `"the cat sat on the mat"` -> last token `"mat"`, 6 tokens of history
- **Truncated** context: `"the cat sat"` -> last token `"sat"`, 3 tokens of history

**Predict:** at block 3, will these two last-position vectors be very similar or clearly different?


In [ ]:
#  Accumulation Tower: last-position richness grows with depth 
torch.manual_seed(1)
tower_model = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=3)
with torch.no_grad():
    _ = tower_model(torch.tensor([TOKEN_IDS]))


def trace_last_position(ids_list):
    """Return the last-position hidden state after each transformer block."""
    tower_model.eval()
    with torch.no_grad():
        token_ids_t = torch.tensor([ids_list], dtype=torch.long)
        S_t = len(ids_list)
        x = tower_model.token_emb(token_ids_t) + tower_model.pe[:S_t]
        causal_m = torch.triu(torch.ones(S_t, S_t, dtype=torch.bool), diagonal=1)
        reps = []
        for block in tower_model.blocks:
            x, _ = block(x, mask=causal_m)
            reps.append(x[0, -1, :].detach().numpy())
    return reps


reps_full  = trace_last_position(TOKEN_IDS)
reps_trunc = trace_last_position(TOKEN_IDS[:3])


def cos_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return float(a @ b)


similarities = [cos_sim(reps_full[d], reps_trunc[d]) for d in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for label, reps, col in [('full (6 tokens)', reps_full, 'royalblue'), ('truncated (3 tokens)', reps_trunc, 'tomato')]:
    norms = [np.linalg.norm(r) for r in reps]
    ax.plot(range(1, 4), norms, 'o-', lw=2, label=label, color=col)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['Block 1', 'Block 2', 'Block 3'])
ax.set_ylabel('L2 norm of last-position vector'); ax.set_title('Representation grows as context accumulates'); ax.legend(fontsize=9)

ax2 = axes[1]
bar_c = ['#5ab4ac' if s > 0.9 else ('#d8b365' if s > 0.7 else 'tomato') for s in similarities]
ax2.bar(range(3), similarities, color=bar_c, alpha=0.9, edgecolor='white', lw=1.2)
ax2.plot(range(3), similarities, 'ko--', lw=1.5, ms=7)
for i, s in enumerate(similarities):
    ax2.text(i, s + 0.01, f'{s:.3f}', ha='center', fontsize=11, fontweight='bold')
ax2.set_xticks([0, 1, 2]); ax2.set_xticklabels(['Block 1', 'Block 2', 'Block 3'])
ax2.set_ylabel('Cosine similarity')
ax2.set_title('Last-position vectors (full vs truncated context)\ndiverge as depth grows', fontsize=10)
ax2.set_ylim(0, 1.1); ax2.axhline(1.0, color='lightgray', ls='--', lw=1)

plt.suptitle('Accumulation Tower: same token ID, different history -> representations diverge with depth',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

for d, s in enumerate(similarities):
    tag = 'still similar' if s > 0.9 else ('diverging' if s > 0.7 else 'very different')
    print(f'  Block {d+1}: cosine similarity = {s:.3f}  ({tag})')
print('  -> Deeper stacks make the last position a richer accumulation point.')


![Three Transformer architecture families: encoder-only (BERT), decoder-only (GPT), encoder-decoder (T5)](images/transformer-architecture-families.png)

---

## Part 13 - Three Architectures: Reader, Writer, Translator

| Architecture | Mask on self-attn | Primary output | Real examples |
| ------------ | ----------------- | -------------- | ------------- |
| **Encoder-only** | None (bidirectional) | Enriched vector per input token | BERT, RoBERTa |
| **Decoder-only** | Causal (lower-tri) | Next-token logits | GPT, LLaMA |
| **Encoder-Decoder** | Enc: none / Dec: causal | Seq2seq translation | T5, BART |


### 13a - Encoder: The Reader (Bidirectional Attention)

An encoder removes the mask entirely. Every token can attend to every other token. `"cat"` at position 1 can immediately see `"mat"` at position 5.

The encoder does **not** predict next tokens. It produces a sequence of enriched context vectors - one per input token.


In [ ]:
#  Part 13a: MiniEncoder - decoder-only twin, minus the mask 


class MiniEncoder(nn.Module):
    """
    Encoder-only transformer (BERT-style).
    Passes mask=None to every TransformerBlock - all tokens see all tokens.
    Output: one enriched d_model-dimensional vector per input token.
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, token_ids, return_attn=False):
        S = token_ids.shape[1]
        x = self.token_emb(token_ids) + self.pe[:S]
        all_attn = []
        for block in self.blocks:
            x, aw = block(x, mask=None)   # None = bidirectional; the only difference
            all_attn.append(aw)
        out = self.norm_out(x)
        if return_attn:
            return out, all_attn
        return out


torch.manual_seed(42)
encoder = MiniEncoder(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
ids_batch = torch.tensor([TOKEN_IDS])
with torch.no_grad():
    enc_out, enc_attns = encoder(ids_batch, return_attn=True)

print(f'Encoder output shape: {tuple(enc_out.shape)}')
print(f'  -> One enriched {D_WORK}-dim vector per token (same shape as decoder output)')
print(f'  -> These are NOT next-token predictions - they are context carriers')
print()
print('The entire MiniEncoder class differs from MiniLM in exactly one place:')
print('  MiniLM      block(x, mask=causal_mask)   # upper triangle -> -inf')
print('  MiniEncoder block(x, mask=None)           # nothing blocked')


#### Predict first - which cells in the heatmap open up?

The decoder's causal heatmap has a hard lower-triangle: `"the"` at [0] attends only to itself.

Now we set `mask=None`. **Predict:**
- How many tokens will `"the"` at [0] attend to now - 1, 3, or 6?
- Will `"the"` at [0] have a non-zero score for `"mat"` at position [5]?


In [ ]:
#  Encoder vs Decoder: SAME MHA weights, SAME input - only the mask differs 
torch.manual_seed(42)
shared_mha = MultiHeadAttention(D_WORK, NUM_HEADS)
causal_mask_cmp = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool), diagonal=1)

with torch.no_grad():
    _, w_bidir      = shared_mha(x_work, mask=None)
    _, w_causal_cmp = shared_mha(x_work, mask=causal_mask_cmp)

bidir_h0  = w_bidir[0, 0].detach().numpy()
causal_h0 = w_causal_cmp[0, 0].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, data, title, cmap in [
    (axes[0], bidir_h0,  'Encoder  (mask=None, bidirectional)\n'
              '"the" at [0] already attends to "cat", "sat", "mat" in layer 1', 'Greens'),
    (axes[1], causal_h0, 'Decoder  (mask=causal_mask, lower triangle only)\n'
              '"the" at [0] sees only itself', 'Oranges'),
]:
    sns.heatmap(data, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
                cbar_kws={'label': 'attention weight'})
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)

plt.suptitle('Same MHA layer, same weights, same input - the mask is the ONLY difference\n'
             'This is the complete implementation difference between Encoder and Decoder',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('Encoder: "the" at [0] attends to all 6 tokens from layer 1.')
print('Decoder: "the" at [0] sees only itself.')
print()
print('Consequence:')
print('  Encoder -> each output holds full bidirectional context')
print('  Decoder -> each output depends ONLY on what came before')


In [ ]:
#  EXERCISE 4 - toggle the mask and watch the heatmap change 
# Flip USE_ENCODER_MASK to False.
# PREDICT first: which cells in row 0 will go to zero?
USE_ENCODER_MASK = True   # set to False to turn the encoder into a decoder

mask_ex4 = None if USE_ENCODER_MASK else causal_mask_cmp
with torch.no_grad():
    _, w_ex4 = shared_mha(x_work, mask=mask_ex4)
w_ex4_h0 = w_ex4[0, 0].detach().numpy()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
mode_label = 'Encoder (mask=None)' if USE_ENCODER_MASK else 'Decoder (causal mask)'
sns.heatmap(w_ex4_h0, ax=ax, annot=True, fmt='.2f',
            cmap='Greens' if USE_ENCODER_MASK else 'Oranges',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
ax.set_title(f'Head 0 attention - {mode_label}')
ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
print(f'Mode: {mode_label}')


#### But wait - why not just pass the encoder output directly as the decoder's starting state?

There are two problems:

**Problem 1 - The causal mask cuts off the source.** The decoder still applies its causal mask.

**Problem 2 - Source and target have different lengths.** You can't stack them as positions.

The solution is **Cross-Attention**: the decoder queries the encoder output as a separate "table" at every decoding step.


### 13b - Cross-Attention: The Bridge

In cross-attention:

$$Q = \text{decoder state} \cdot W_Q \qquad K, V = \text{encoder output} \cdot W_K, W_V$$

The decoder asks: *"Given what I have generated so far ($Q$), which part of the source text ($K$) is most relevant, and what should I extract from it ($V$)?"*


In [ ]:
#  CrossAttention: Q from decoder, K/V from encoder 


class CrossAttention(nn.Module):
    """
    Cross-attention layer.
    Q  <- decoder's current state
    K  <- encoder's output
    V  <- encoder's output
    No mask on the encoder dimension - the decoder can attend to ANY source position.
    """

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, decoder_x, encoder_kv):
        """
        decoder_x : (B, tgt_len, d_model)
        encoder_kv: (B, src_len, d_model)
        Score matrix: (B, heads, tgt_len, src_len)  - no mask applied.
        """
        B, tgt_len, _ = decoder_x.shape
        src_len = encoder_kv.shape[1]
        Q_ca = self.W_Q(decoder_x).reshape(B, tgt_len, self.n_heads, self.d_head).transpose(1, 2)
        K_ca = self.W_K(encoder_kv).reshape(B, src_len, self.n_heads, self.d_head).transpose(1, 2)
        V_ca = self.W_V(encoder_kv).reshape(B, src_len, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q_ca @ K_ca.transpose(-2, -1)) / math.sqrt(self.d_head)
        cross_w = torch.softmax(scores, dim=-1)   # (B, H, tgt, src)
        out = cross_w @ V_ca
        out = out.transpose(1, 2).reshape(B, tgt_len, self.d_model)
        return self.W_O(out), cross_w


#  Quick demo: one decoder query token attending to six encoder positions 
torch.manual_seed(5)
cross_demo = CrossAttention(D_WORK, NUM_HEADS)
dec_state_demo = enc_out[:, :1, :]
with torch.no_grad():
    ca_out_demo, ca_w_demo = cross_demo(dec_state_demo, enc_out)

print('Cross-attention shapes:')
print(f'  Decoder query  : {tuple(dec_state_demo.shape)}  (1 decoder token)')
print(f'  Encoder K/V    : {tuple(enc_out.shape)}  (6 source tokens)')
print(f'  Score matrix   : {tuple(ca_w_demo.shape)}')
print()
print('The single decoder query scored all 6 encoder positions.')
print('Which source token it attends to is entirely learned by the loss.')


### 13c - Encoder-Decoder: The Translator

A full encoder-decoder wires both halves together with cross-attention in every decoder block. We train it on a toy task - **reversing a three-word phrase** - to force cross-attention to do real work.

```
Encoder: ["the", "cat", "sat"]  ->  [ enriched vectors: V_the, V_cat, V_sat ]
Decoder: [<BOS>]  -> "sat" ;  [<BOS>, sat]  -> "cat" ;  [<BOS>, sat, cat]  -> "the"
```


In [ ]:
#  DecoderBlockWithCrossAttn + MiniEncoderDecoder 


class DecoderBlockWithCrossAttn(nn.Module):
    """
    Full decoder block - three sub-layers:
      1. Causal self-attention  - the decoder looks at its own generated tokens
      2. Cross-attention        - the decoder queries the encoder's source map
      3. Feed-forward           - per-token nonlinear transformation
    """

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.cross_attn = CrossAttention(d_model, n_heads)
        self.norm3 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, encoder_output, causal_mask=None):
        sa_out, sa_w = self.self_attn(self.norm1(x), mask=causal_mask)
        x = x + sa_out
        ca_out, ca_w = self.cross_attn(self.norm2(x), encoder_output)
        x = x + ca_out
        x = x + self.ffn(self.norm3(x))
        return x, sa_w, ca_w


class MiniEncoderDecoder(nn.Module):
    """
    Encoder-Decoder transformer (T5 / original-Transformer style).
    Encoder : bidirectional - produces a frozen source map (K, V for cross-attn)
    Decoder : causal self-attn + cross-attn at every block - LM head
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab_size, d_model)
        pe = sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)
        self.enc_blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.enc_norm = nn.LayerNorm(d_model, eps=1e-5)
        self.dec_blocks = nn.ModuleList([
            DecoderBlockWithCrossAttn(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model, eps=1e-5)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def encode(self, src_ids):
        S_enc = src_ids.shape[1]
        x = self.emb(src_ids) + self.pe[:S_enc]
        for block in self.enc_blocks:
            x, _ = block(x, mask=None)
        return self.enc_norm(x)

    def forward(self, src_ids, tgt_ids):
        """
        src_ids: (B, src_len) - what the encoder reads
        tgt_ids: (B, tgt_len) - decoder input shifted right (teacher-forced)
        Returns logits (B, tgt_len, vocab_size) and all cross-attn weight tensors.
        """
        enc_out_s2s = self.encode(src_ids)
        T_dec = tgt_ids.shape[1]
        x = self.emb(tgt_ids) + self.pe[:T_dec]
        causal_mask_s2s = torch.triu(torch.ones(T_dec, T_dec, dtype=torch.bool), diagonal=1)
        causal_mask_s2s = causal_mask_s2s.to(x.device)
        all_ca_w = []
        for block in self.dec_blocks:
            x, _, ca_w = block(x, enc_out_s2s, causal_mask=causal_mask_s2s)
            all_ca_w.append(ca_w)
        x = self.dec_norm(x)
        logits = self.lm_head(x)
        return logits, all_ca_w


torch.manual_seed(42)
seq2seq_demo = MiniEncoderDecoder(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
print('MiniEncoderDecoder architecture:')
print(f'  Encoder: {len(seq2seq_demo.enc_blocks)} x TransformerBlock  (mask=None, bidirectional)')
print(f'  Decoder: {len(seq2seq_demo.dec_blocks)} x DecoderBlockWithCrossAttn')
print(f'           -> self-attn (causal) + cross-attn + FFN')
print(f'  Shared vocab: {VOCAB_SIZE} tokens')


In [ ]:
#  Toy training data: reverse a 3-word phrase 
# Source:  ["the", "cat", "sat"]
# Target:  ["sat", "cat", "the", "<EOS>"]  (decoder input = ["<BOS>"] + target[:-1])

BOS_ID, EOS_ID = VOCAB['<BOS>'], VOCAB['<EOS>']

source_phrases_rev = [
    ['the', 'cat', 'sat'], ['the', 'dog', 'ran'],
    ['a',   'big', 'cat'], ['the', 'cat', 'ran'],
    ['a',   'dog', 'sat'], ['the', 'big', 'dog'],
]

REV_DATA = []
for phrase in source_phrases_rev:
    src_ids_r = [VOCAB[w] for w in phrase]
    rev_ids   = [VOCAB[w] for w in reversed(phrase)]
    tgt_in_r  = [BOS_ID] + rev_ids
    tgt_out_r = rev_ids + [EOS_ID]
    REV_DATA.append((src_ids_r, tgt_in_r, tgt_out_r))

print(f'Reversal task - {len(REV_DATA)} training pairs:')
for src_r, _, tout_r in REV_DATA[:3]:
    print(f'  {[IDX2WORD[i] for i in src_r]}  ->  {[IDX2WORD[i] for i in tout_r]}')
print('  ...')


def build_rev_batch(data, pad_id=0):
    src_max = max(len(s) for s, _, _ in data)
    tgt_max = max(len(t) for _, t, _ in data)
    srcs, tins, touts = [], [], []
    for s, ti, to in data:
        srcs.append(s  + [pad_id] * (src_max - len(s)))
        tins.append(ti + [pad_id] * (tgt_max - len(ti)))
        touts.append(to + [pad_id] * (tgt_max - len(to)))
    return (torch.tensor(srcs,  dtype=torch.long),
            torch.tensor(tins,  dtype=torch.long),
            torch.tensor(touts, dtype=torch.long))


src_rev, tgt_in_rev, tgt_out_rev = build_rev_batch(REV_DATA)
print(f'Batch shapes - src: {tuple(src_rev.shape)}  '
      f'tgt_in: {tuple(tgt_in_rev.shape)}  tgt_out: {tuple(tgt_out_rev.shape)}')


In [ ]:
#  Train the encoder-decoder on the reversal task 
torch.manual_seed(42)
seq2seq = MiniEncoderDecoder(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
optimizer_s2s = torch.optim.Adam(seq2seq.parameters(), lr=5e-3)
loss_fn_s2s = nn.CrossEntropyLoss(ignore_index=0)

EPOCHS_S2S = 500
s2s_loss_h, s2s_acc_h = [], []

for epoch in range(EPOCHS_S2S):
    seq2seq.train()
    optimizer_s2s.zero_grad()
    logits_s2s, _ = seq2seq(src_rev, tgt_in_rev)
    loss_s2s = loss_fn_s2s(logits_s2s.reshape(-1, VOCAB_SIZE), tgt_out_rev.reshape(-1))
    loss_s2s.backward()
    torch.nn.utils.clip_grad_norm_(seq2seq.parameters(), 1.0)
    optimizer_s2s.step()

    if (epoch + 1) % 25 == 0:
        with torch.no_grad():
            preds_s2s = torch.argmax(logits_s2s, dim=-1)
            pad_mask_s2s = (tgt_out_rev != 0)
            correct = (preds_s2s == tgt_out_rev) & pad_mask_s2s
            acc_s2s = correct.float().sum().item() / pad_mask_s2s.float().sum().item()
        s2s_loss_h.append(loss_s2s.item())
        s2s_acc_h.append(acc_s2s)
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1:4d}  loss={loss_s2s.item():.4f}  acc={acc_s2s:.2%}')

print('\nTraining complete.  Greedy decoding results:')
seq2seq.eval()
for src_r, tin_r, tout_r in REV_DATA:
    with torch.no_grad():
        lgts, _ = seq2seq(
            torch.tensor([src_r],  dtype=torch.long),
            torch.tensor([tin_r],  dtype=torch.long),
        )
    pred_ids = torch.argmax(lgts[0], dim=-1).tolist()
    src_w  = [IDX2WORD[i] for i in src_r]
    pred_w = [IDX2WORD[i] for i in pred_ids]
    gold_w = [IDX2WORD[i] for i in tout_r]
    ok = '[ok]' if pred_ids == tout_r else '[xx]'
    print(f'  {ok}  source={src_w}  pred={pred_w}  gold={gold_w}')


#### Predict first - draw the cross-attention map

Source: `["the", "cat", "sat"]` -> reversed target: `["sat", "cat", "the"]`.

| Decoder step | Generating | Highest source attention should be at? |
| ------------ | ---------- | --------------------------------------- |
| Step 0 (`<BOS>` -> "sat") | "sat" | source[ ? ] |
| Step 1 (`"sat"` -> "cat") | "cat" | source[ ? ] |
| Step 2 (`"sat cat"` -> "the") | "the" | source[ ? ] |


In [ ]:
#  Cross-attention heatmap: decoder reading the encoder blueprint 
test_src_s2s = [VOCAB['the'], VOCAB['cat'], VOCAB['sat']]
test_tgt_s2s = [BOS_ID, VOCAB['sat'], VOCAB['cat']]

seq2seq.eval()
with torch.no_grad():
    logits_vis, ca_vis = seq2seq(
        torch.tensor([test_src_s2s], dtype=torch.long),
        torch.tensor([test_tgt_s2s], dtype=torch.long),
    )

src_lbls = [IDX2WORD[i] for i in test_src_s2s]
dec_lbls = ['<BOS>->sat', 'sat->cat', 'cat->the']
n_dec_layers = len(seq2seq.dec_blocks)
fig, axes = plt.subplots(1, n_dec_layers, figsize=(6 * n_dec_layers, 4.2))
if n_dec_layers == 1:
    axes = [axes]

for layer_i, (ax, ca_w) in enumerate(zip(axes, ca_vis)):
    mean_ca = ca_w[0].mean(dim=0).detach().numpy()   # avg heads -> (tgt, src)
    sns.heatmap(mean_ca, ax=ax, annot=True, fmt='.2f',
                cmap='YlOrRd', xticklabels=src_lbls, yticklabels=dec_lbls,
                linewidths=0.6, vmin=0, vmax=1, cbar_kws={'label': 'cross-attn weight'})
    ax.set_title(f'Decoder layer {layer_i}  (mean over {NUM_HEADS} heads)')
    ax.set_xlabel('Source position (encoder output)')
    ax.set_ylabel('Decoder step -> predicted token')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle(f'Cross-attention: decoder querying the encoder  |  source: {src_lbls}',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('If reversal is learned, row 0 (->"sat") should attend most to source[2]="sat".')
print('Row 1 (->"cat") should attend most to source[1]="cat".  Etc.')
print('The cross-attention map IS the learned "look at the right source position" rule.')


### 13d - Why the Industry Moved to Decoder-Only

| Issue | Encoder-Decoder | Decoder-Only |
| ----- | --------------- | ------------ |
| Training data | Needs paired input/output sequences | Eats *any* raw text |
| KV cache at inference | Two separate caches | One growing cache |
| Information routing | Must compress source through cross-attention | Implicit in early layers |
| Scaling | Complex orchestration | Simple stack |

#### What is a "KV cache", actually?

The autoregressive loop in Part 9 (`generate_next`) re-encodes the entire growing context from scratch at every new token, recomputing K and V for every earlier position again even though those tokens haven't changed. A KV cache is the optimisation of not redoing that work: every past token's K and V vectors are computed once and stored, so each new step only computes Q/K/V for the one new token and reuses the cached K/V for everything before it. An encoder-decoder model needs two such caches (one for the fixed encoder output, one for the growing decoder prefix); a decoder-only model needs only one (the growing prefix) - exactly the "two caches vs. one growing cache" row above.

Not implemented here: the toy generation loop is short enough (a handful of tokens) that recomputation is free. A real inference server needs the cache so that per-token latency stays roughly constant instead of growing with sequence length.


In [ ]:
#  Architecture comparison: parameter cost and capability table 
torch.manual_seed(0)

_enc_cmp    = MiniEncoder(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
_dec_cmp    = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
_encdec_cmp = MiniEncoderDecoder(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)

_dummy6 = torch.tensor([TOKEN_IDS])
_dummy3 = torch.tensor([TOKEN_IDS[:3]])
with torch.no_grad():
    _enc_cmp(_dummy6)
    _dec_cmp(_dummy6)
    _encdec_cmp(_dummy6, _dummy3)

p_enc_cmp    = sum(p.numel() for p in _enc_cmp.parameters())
p_dec_cmp    = sum(p.numel() for p in _dec_cmp.parameters())
p_encdec_cmp = sum(p.numel() for p in _encdec_cmp.parameters())

rows_cmp = [
    ('',                'Encoder-Only',  'Decoder-Only',     'Encoder-Decoder'),
    ('Self-attn mask',  'None (bidir)',  'Causal',           'Enc: none / Dec: causal'),
    ('Cross-attn',      'No',            'No',               f'{len(_encdec_cmp.dec_blocks)} layers'),
    ('Parameters',      f'{p_enc_cmp:,}', f'{p_dec_cmp:,}', f'{p_encdec_cmp:,}'),
    ('Training data',   'Labelled corpus', 'Any raw text',  'Paired sequences'),
    ('Real examples',   'BERT, RoBERTa',   'GPT, LLaMA',    'T5, BART'),
]

col_w = [20, 18, 20, 26]
sep = '  ' + '-' * (sum(col_w) + 6)
print(sep)
for i, row in enumerate(rows_cmp):
    line = '  ' + '  '.join(f'{c:<{w}}' for c, w in zip(row, col_w))
    print(line)
    if i == 0:
        print(sep)
print(sep)
print()
overhead = p_encdec_cmp - p_dec_cmp
print(f'Cross-attention overhead : {overhead:+,} params  '
      f'(+{overhead/p_dec_cmp*100:.1f}% vs decoder-only at same depth/width)')
print()
print('At GPT-3 scale (175B parameters), that overhead is non-trivial -')
print('one reason the industry consolidated around decoder-only for general LLMs.')


---

## Part 14 - Cracking Open distilgpt2

Now we scale from our toy model to a real one: **DistilGPT-2** with:

- 6 transformer blocks
- 12 attention heads per block
- d_model = 768
- ~82M parameters

We will:
1. Load the model and inspect its architecture
2. Run a forward pass with `output_attentions=True` to capture all 6 layers x 12 heads
3. Visualise the attention patterns
4. Plot the next-token probability distribution


In [ ]:
#  Load DistilGPT-2 (PyTorch backend) 
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt_tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
gpt2 = GPT2LMHeadModel.from_pretrained('distilgpt2')
gpt2.eval()

cfg = gpt2.config
print('=== DistilGPT-2 Architecture ===')
print(f'  n_layer         : {cfg.n_layer}')
print(f'  n_head          : {cfg.n_head}')
print(f'  n_embd (d_model): {cfg.n_embd}')
print(f'  vocab_size      : {cfg.vocab_size}')
print(f'  n_positions     : {cfg.n_positions}  (max context)')
print(f'  wpe (pos. emb.) : {tuple(gpt2.transformer.wpe.weight.shape)}  (learned, not sinusoidal!)')
print()
n_params_gpt = sum(p.numel() for p in gpt2.parameters())
print(f'  Total parameters: {n_params_gpt:,}  (~{n_params_gpt/1e6:.1f}M)')
print()
print('Top-level modules:')
for name, module in gpt2.named_children():
    print(f'  {name}: {module.__class__.__name__}')
print()
print('First transformer block sub-modules:')
block0 = gpt2.transformer.h[0]
for name, mod in block0.named_children():
    print(f'  h[0].{name}: {mod.__class__.__name__}')
print()
print('Real BPE tokenisation, e.g. on an unfamiliar word:')
for rare_word in ['internationalization', 'zzzflorptastic']:
    pieces = gpt_tokenizer.tokenize(rare_word)
    print(f'  {rare_word!r:<24} -> {len(pieces)} piece(s): {pieces}')
print('  -> Unlike our one-word-one-token toy scheme, BPE splits unfamiliar words into')
print('     smaller, previously-seen sub-word chunks, so the vocabulary never needs an')
print('     <UNK> token for a whole word it has not seen before.')


In [ ]:
#  Run inference with all attentions captured 
PROMPT = 'The cat sat on the'
inputs = gpt_tokenizer(PROMPT, return_tensors='pt')
input_ids = inputs['input_ids']

gpt_tokens = [gpt_tokenizer.decode([int(t)]) for t in input_ids[0]]
print(f'Prompt       : {PROMPT!r}')
print(f'GPT-2 tokens : {gpt_tokens}')
print(f'Token IDs    : {input_ids[0].tolist()}')
print()

with torch.no_grad():
    out = gpt2(**inputs, output_attentions=True)

print(f'Number of attention layers returned: {len(out.attentions)}')
print(f'Each layer shape: {tuple(out.attentions[0].shape)}')
print(f'  (batch=1, n_heads=12, seq={len(gpt_tokens)}, seq={len(gpt_tokens)})')
print()

next_logits = out.logits[0, -1, :]
next_probs = torch.softmax(next_logits, dim=-1)
top10 = torch.topk(next_probs, k=10)

print(f'Top-10 next token predictions after "{PROMPT}":')
for rank, (tid, prob) in enumerate(zip(top10.indices.tolist(), top10.values.tolist()), 1):
    tok = gpt_tokenizer.decode([int(tid)])
    print(f'  {rank:2d}. {tok!r:<20} {float(prob):.4f}')


In [ ]:
#  Attention pattern visualisation - all 6 layers, mean over heads 
n_layers_gpt = len(out.attentions)
seq_len_gpt = len(gpt_tokens)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for layer_idx, (layer_attn, ax) in enumerate(zip(out.attentions, axes)):
    mean_attn = layer_attn[0].mean(dim=0).detach().numpy()
    sns.heatmap(mean_attn, ax=ax, cmap='viridis',
                xticklabels=gpt_tokens, yticklabels=gpt_tokens,
                linewidths=0.3, cbar_kws={'label': 'attn weight'})
    ax.set_title(f'Layer {layer_idx}  (mean over 12 heads)', fontsize=9)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('DistilGPT-2: mean attention patterns across all 6 layers',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
#  Head diversity - compare individual heads in layer 0 
layer0_attn = out.attentions[0][0].detach().numpy()   # (12, seq, seq)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for h, ax in enumerate(axes):
    sns.heatmap(layer0_attn[h], ax=ax, cmap='Blues',
                xticklabels=gpt_tokens, yticklabels=gpt_tokens,
                linewidths=0.3, annot=True, fmt='.2f', annot_kws={'size': 8})
    ax.set_title(f'Layer 0  Head {h}', fontsize=9)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle('DistilGPT-2 Layer 0: first 4 heads show diverse specialisation',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
#  Next-token probability bar chart 
top_n = 15
topk_gpt = torch.topk(next_probs, k=top_n)
top_tokens_gpt = [gpt_tokenizer.decode([int(t)]) for t in topk_gpt.indices.tolist()]
top_probs_gpt = topk_gpt.values.detach().numpy()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['gold'] + ['steelblue'] * (top_n - 1)
ax.barh(top_tokens_gpt[::-1], top_probs_gpt[::-1], color=colors[::-1], alpha=0.85)
ax.set_xlabel('Probability')
ax.set_title(f'DistilGPT-2 next-token probabilities\nPrompt: "{PROMPT}"',
             fontsize=11)
for bar, prob in zip(ax.patches, top_probs_gpt[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{prob:.4f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

print(f'Top prediction: {top_tokens_gpt[0]!r}  (prob={top_probs_gpt[0]:.4f})')


In [ ]:
#  Full autoregressive generation with distilgpt2 


def gpt2_generate(prompt: str, max_new_tokens: int = 15, temperature: float = 0.8, top_k: int = 50):
    """Generate tokens one at a time, printing each step."""
    gpt2.eval()
    ids = gpt_tokenizer.encode(prompt, return_tensors='pt')
    print(f'Prompt: {prompt!r}')
    print(f'Starting IDs: {ids[0].tolist()}')
    print()

    generated = ids
    for step in range(max_new_tokens):
        with torch.no_grad():
            out_step = gpt2(generated)
        logits_step = out_step.logits[0, -1, :].clone()

        # Temperature + top-k filtering
        logits_step = logits_step / max(temperature, 1e-8)
        if top_k > 0:
            top_vals, top_idx = torch.topk(logits_step, k=top_k)
            filtered = torch.full_like(logits_step, float('-inf'))
            filtered.scatter_(0, top_idx, top_vals)
            logits_step = filtered

        probs_step = torch.softmax(logits_step, dim=-1)
        next_id = int(torch.multinomial(probs_step, num_samples=1).item())
        next_tok = gpt_tokenizer.decode([next_id])

        print(f'  Step {step+1:2d} - token ID {next_id:5d}  {next_tok!r:<20}  '
              f'p={probs_step[next_id].item():.4f}')

        generated = torch.cat([generated, torch.tensor([[next_id]])], dim=1)

        if next_id == gpt_tokenizer.eos_token_id:
            print('  [EOS - stopping]')
            break

    final_text = gpt_tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
    print()
    print(f'Final: {final_text!r}')
    return final_text


gpt2_generate('The cat sat on the', max_new_tokens=12, temperature=0.7)


---

## What This Notebook Covered (and What It Didn't)

Before the final recap below, here is the honest three-tier breakdown of the full topic space a complete "how Transformers work" treatment would include, and exactly where this notebook's actual code lands on each one.

**Tier 1 - Implemented and demonstrated:** essentially everything above, Parts 1-14 - see the completed roadmap table in the Summary immediately below for the full list.

**Tier 2 - Explained but not fully implemented** (accurate explanation, no full toy-then-real demo):

- **KV-caching** - explained in plain English right after the encoder-decoder vs. decoder-only comparison table, tied directly to the recomputation this notebook's own `generate_next` loop does at every step, but no actual cache tensors are built.
- **Pre-LN vs. Post-LN placement** - this notebook uses and names Pre-LN; Post-LN is named as the historical alternative but not built side-by-side for comparison.

**Tier 3 - Named but out of scope** (acknowledged, with a one-line reason):

- **Learned absolute positional embeddings** and **ALiBi** - two more positional-encoding schemes in the same family as sinusoidal PE / RoPE, omitted to keep the sinusoidal -> RoPE arc as the notebook's throughline. (DistilGPT-2's real `wpe` table is inspected in Part 14 as a concrete example of the learned variant, without re-deriving it from scratch.)
- **Dropout** - a standard regulariser, omitted because the toy models are never at risk of overfitting a six-sentence corpus.
- **Model scaling laws** - a training-economics topic (compute/data/params trade-offs), orthogonal to the architecture mechanics this notebook focuses on.
- **Nucleus (top-p) sampling and beam search** - greedy/temperature/top-k already demonstrate the sampling-strategy idea; two more decoding algorithms wouldn't teach a new mechanism.
- **Efficient-attention variants** (sliding-window, sparse attention, FlashAttention, multi-query / grouped-query attention) - production-scale engineering optimisations of the exact O(n^2) attention already built here, not a different mechanism.
- **Explicit padding/attention masks for batched variable-length input** - this notebook's training loops use fixed-length batches or loss-side masking (`ignore_index`); a production implementation would also mask padded positions out of attention itself.


---

## Summary - The Complete Transformer Journey

We've traced every component from raw words to generated tokens:

| Step | Component | What happens |
| ---- | --------- | ------------ |
| 1  | **Tokeniser** | Text -> integer IDs |
| 2  | **Token Embedding** | IDs -> dense vectors in semantic space |
| 3  | **Positional Encoding** | Add position signal (sin/cos or RoPE rotation) |
| 4  | **Q/K/V Projection** | Three learned views of each token vector |
| 5  | **Scaled Dot-Product Attention** | Soft dictionary lookup - compute relevance scores |
| 6  | **Multi-Head Attention** | H parallel attention views concatenated |
| 7  | **Feed-Forward Network** | Per-token nonlinear transformation |
| 8  | **LayerNorm + Residuals** | Stabilise activations, guarantee gradient flow |
| 9  | **Repeat x L** | Stack L transformer blocks |
| 10 | **LM Head** | Project to vocabulary -> logits -> softmax -> distribution |
| 11 | **W_V Relevance Filter** | W_V extracts the task-specific payload |
| 12 | **Causal Triangle** | Position n accumulates n+1 tokens; depth chains richness |
| 13 | **Encoder Architecture** | mask=None gives bidirectional context |
| 14 | **Cross-Attention** | Q from decoder, K/V from frozen encoder |
| 15 | **Encoder-Decoder** | Source encoded once; decoder cross-attends at every step |
| 16 | **Architecture Comparison** | Decoder-only won at scale; encoder stays essential |
| 17 | **GPT-2 Internals** | A real model, cracked open |
| 18 | **Autoregressive loop** | Sample next token -> append -> repeat |

### Key insights to keep

- **RoPE** encodes position by rotating Q and K; only the relative gap survives in dot-products
- **Attention is O(n^2)** in sequence length - this is why long-context models are expensive
- **Multi-head attention** lets each head specialise on a different relationship type
- **Residual connections** make depth practical - gradients always have a direct path home
- **Temperature** is the single most intuitive control knob at inference time
- **W_V is a relevance filter** - it extracts the task-specific slice of each token's information
- **The causal triangle means depth compounds** - position n at layer L has processed a chain of enriched representations from all n earlier tokens
- **Encoder = mask removed** - `mask=None` gives every token a full-sentence view from layer 1
- **Cross-attention decouples source and target** - Q from the decoder re-queries a frozen encoder map at every step; no compression bottleneck
- **Decoder-only scales cleanly** - no paired data needed
